# Retrieval and Ranking

**Stage 1** — Bi-encoder dense retrieval via FAISS
- Top-5 standalone result -> `retrieval_biencoder_top5.jsonl`
- Top-20 candidate pool -> `retrieval_biencoder_top20.jsonl` (input for Stage 2 only)

**Stage 2** — Cross-encoder reranking (reads top-20, outputs top-5)
- Reranked result -> `retrieval_reranked_top5.jsonl`

In [1]:
!pip install sentence-transformers

In [2]:
import json
import os
import time
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from typing import List, Dict

d:\app\miniconda\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Config
BENCHMARK_PATH = "../data/benchmark.json"
EMBEDDED_PATH = "../embedding/output/chunks_fixed300_overlap0/embedded_minilm.jsonl"
FAISS_PATH = "../embedding/output/chunks_fixed300_overlap0/embedded_minilm.faiss"
MODEL_NAME = "all-MiniLM-L6-v2"

RESULTS_DIR = "results/"
os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
# Load benchmark
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    benchmark = json.load(f)
print(f"Loaded {len(benchmark)} benchmark queries")

# Load FAISS index + corpus lines
index = faiss.read_index(FAISS_PATH)
with open(EMBEDDED_PATH, "r", encoding="utf-8") as f:
    corpus_lines = f.readlines()
assert len(corpus_lines) == index.ntotal
print(f"FAISS index: {index.ntotal} vectors, dim={index.d}")

# Load bi-encoder
bi_encoder = SentenceTransformer(MODEL_NAME)
print(f"Bi-encoder: {MODEL_NAME}")

Loaded 10 benchmark queries
FAISS index: 1838 vectors, dim=384


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 595.04it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bi-encoder: all-MiniLM-L6-v2


In [5]:
def retrieve_biencoder(
    queries: List[Dict],
    index: faiss.Index,
    corpus_lines: List[str],
    model: SentenceTransformer,
    top_n: int,
) -> List[Dict]:
    """Bi-encoder retrieval for all queries. Returns top-n candidates per query."""
    query_texts = [q["query"] for q in queries]
    query_embs = model.encode(
        query_texts, normalize_embeddings=True, show_progress_bar=True
    ).astype(np.float32)

    scores_all, indices_all = index.search(query_embs, top_n)

    results = []
    for q, scores, indices in zip(queries, scores_all, indices_all):
        retrieved = []
        for rank, (score, idx) in enumerate(zip(scores, indices), start=1):
            if idx == -1:
                continue
            chunk = json.loads(corpus_lines[idx])
            retrieved.append({
                "rank": rank,
                "chunk_id": chunk["chunk_id"],
                "score": round(float(score), 6),
                "text": chunk["text"],
            })
        results.append({
            "query_id": q["query_id"],
            "query": q["query"],
            "retrieved": retrieved,
        })
    return results


def save_results(results: List[Dict], path: str):
    with open(path, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(results)} queries to {path}")

---

## Stage 1A: Top-5 (standalone)

Final retrieval result when no reranker is used. Ready for prompting and evaluation.

In [6]:
t0 = time.time()
results_top5 = retrieve_biencoder(benchmark, index, corpus_lines, bi_encoder, top_n=5)
print(f"Done: {len(results_top5)} queries, {time.time()-t0:.2f}s")

save_results(results_top5, os.path.join(RESULTS_DIR, "retrieval_biencoder_top5.jsonl"))

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.75it/s]

Done: 10 queries, 0.24s
Saved 10 queries to results/retrieval_biencoder_top5.jsonl


In [7]:
# Inspect first query
sample = results_top5[0]
print(f"Query: {sample['query']}\n")
for r in sample["retrieved"]:
    print(f"  #{r['rank']}  score={r['score']:.4f}  {r['chunk_id']}")
    print(f"       {r['text'][:100]}...\n")

Query: What is sushi made of?

  #1  score=0.7450  wiki_0259_chunk_0
       Sushi (すし, 寿司, 鮨, 鮓; pronounced [sɯɕiꜜ] or [sɯꜜɕi] ) is a traditional Japanese dish made with vinega...

  #2  score=0.6993  wiki_0259_chunk_25
       of shellfish (abalone, prawn, scallop) are the most popular seafoods in sushi. Oysters are less comm...

  #3  score=0.6874  wiki_0259_chunk_27
       asa veitchii (熊笹, kuma-zasa) plants, respectively. Using actual leaves had the added benefit of rele...

  #4  score=0.6704  wiki_0259_chunk_7
       some other flavoring". There is an earlier mention of sushi in James Hepburn's Japanese–English dict...

  #5  score=0.6605  wiki_0259_chunk_22
       酢飯), shari (舎利), or gohan (ご飯)) is a preparation of white, short-grained, Japanese rice mixed with a...



---

## Stage 1B: Top-20 (for Stage 2 reranker)


It is a wider candidate pool so the cross-encoder reranker can pick better top-5 from 20 candidates instead of just reordering 5.

In [ ]:
t0 = time.time()
results_top20 = retrieve_biencoder(benchmark, index, corpus_lines, bi_encoder, top_n=20)
print(f"Done: {len(results_top20)} queries, {time.time()-t0:.2f}s")

save_results(results_top20, os.path.join(RESULTS_DIR, "retrieval_biencoder_top20.jsonl"))

---
---

## Stage 2: Cross-Encoder Reranking

Read the top-20 candidates above, rerank with a cross-encoder, output top-5.
